# Results Visualization Notebook

**Purpose**: Visualize model predictions, confidence distributions, and summary statistics

**Outputs**: Prediction visualizations, confidence plots, summary statistics

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import yaml

sys.path.insert(0, '.')

from src.models.factory import ModelFactory
from src.data.dataloader import create_test_dataloader

# Configuration
config_path = 'configs/experiments/baseline.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model
model = ModelFactory.create(
    name=config['model']['name'],
    num_classes=config['model']['num_classes'],
    hidden_features=config['model']['hidden_features']
)

checkpoint_path = Path('artifacts/checkpoints/last.pt')
if checkpoint_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"✓ Model loaded")
else:
    print("⚠️  No checkpoint found")

model.to(device)
model.eval()

# Load test data
test_loader = create_test_dataloader(
    data_dir=config['data']['data_dir'],
    batch_size=64
)

print(f"✓ Data loaded: {len(test_loader)} batches")

In [ ]:
# Generate predictions with probabilities
all_images = []
all_preds = []
all_labels = []
all_confidences = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        
        outputs = model(X_batch)
        probs = torch.softmax(outputs, dim=1)
        confidences, preds = torch.max(probs, dim=1)
        
        all_images.extend(X_batch.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.numpy())
        all_confidences.extend(confidences.cpu().numpy())

all_images = np.array(all_images)
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_confidences = np.array(all_confidences)

accuracy = (all_preds == all_labels).mean()
print(f"✓ Predictions generated")
print(f"  Total samples: {len(all_preds)}")
print(f"  Accuracy: {accuracy * 100:.2f}%")
print(f"  Avg confidence: {all_confidences.mean():.4f}")

In [ ]:
# Visualize correct and incorrect predictions
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

# Correct predictions
correct_mask = all_preds == all_labels
correct_indices = np.where(correct_mask)[0][:5]

for idx, ax_idx in enumerate(range(5)):
    sample_idx = correct_indices[idx]
    ax = axes[0, ax_idx]
    ax.imshow(all_images[sample_idx], cmap='gray')
    ax.set_title(f'✓ {all_labels[sample_idx]}\n{all_confidences[sample_idx]:.3f}',
                fontsize=10, fontweight='bold', color='green')
    ax.axis('off')

# Incorrect predictions
incorrect_mask = all_preds != all_labels
if incorrect_mask.sum() > 0:
    incorrect_indices = np.where(incorrect_mask)[0][:5]
    for idx, ax_idx in enumerate(range(5)):
        if idx < len(incorrect_indices):
            sample_idx = incorrect_indices[idx]
            ax = axes[1, ax_idx]
            ax.imshow(all_images[sample_idx], cmap='gray')
            ax.set_title(f'✗ {all_labels[sample_idx]} → {all_preds[sample_idx]}\n{all_confidences[sample_idx]:.3f}',
                        fontsize=10, fontweight='bold', color='red')
            ax.axis('off')
        else:
            axes[1, ax_idx].axis('off')
else:
    for ax in axes[1, :]:
        ax.axis('off')

plt.suptitle('Sample Predictions: Correct vs Incorrect', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/assets/figures/06_prediction_examples.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Prediction examples saved")

In [ ]:
# Confidence analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Overall confidence distribution
axes[0].hist(all_confidences, bins=50, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(all_confidences.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {all_confidences.mean():.4f}')
axes[0].set_xlabel('Confidence', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Confidence Distribution (All)', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3, axis='y')

# Separate correct/incorrect
correct_conf = all_confidences[all_preds == all_labels]
incorrect_conf = all_confidences[all_preds != all_labels] if incorrect_mask.sum() > 0 else []

axes[1].hist(correct_conf, bins=50, alpha=0.6, label='Correct', color='green', edgecolor='black')
if len(incorrect_conf) > 0:
    axes[1].hist(incorrect_conf, bins=50, alpha=0.6, label='Incorrect', color='red', edgecolor='black')
axes[1].set_xlabel('Confidence', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].set_title('Confidence Distribution (Correct vs Incorrect)', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('notebooks/assets/figures/07_confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Confidence distribution visualization saved")

In [ ]:
# Summary statistics
summary = pd.DataFrame({
    'Metric': [
        'Total Samples',
        'Correct Predictions',
        'Incorrect Predictions',
        'Accuracy',
        'Error Rate',
        'Avg Confidence (All)',
        'Avg Confidence (Correct)',
        'Avg Confidence (Incorrect)',
        'Min Confidence',
        'Max Confidence'
    ],
    'Value': [
        f"{len(all_preds):,}",
        f"{(all_preds == all_labels).sum():,}",
        f"{(all_preds != all_labels).sum():,}",
        f"{accuracy * 100:.2f}%",
        f"{(1 - accuracy) * 100:.2f}%",
        f"{all_confidences.mean():.6f}",
        f"{correct_conf.mean():.6f}",
        f"{incorrect_conf.mean():.6f}" if len(incorrect_conf) > 0 else "N/A",
        f"{all_confidences.min():.6f}",
        f"{all_confidences.max():.6f}"
    ]
})

print("\n" + "="*60)
print("📊 Results Visualization Summary")
print("="*60)
print(summary.to_string(index=False))
print("="*60)

# Save summary
summary.to_csv('notebooks/assets/tables/05_visualization_summary.csv', index=False)
print("\n✓ Summary saved to CSV")
print("\n✅ Results visualization complete!")